In [1]:
import pandas as pd

titanic = pd.read_csv("../data/titanic_propre.csv")

ports = pd.DataFrame({
    "embark_town": ["Southampton", "Cherbourg", "Queenstown"],
    "pays": ["Angleterre", "France", "Irlande"],
})
ports

,embark_town,pays
0,Southampton,Angleterre
1,Cherbourg,France
2,Queenstown,Irlande


In [2]:
titanic_pays = titanic.merge(ports, on="embark_town", how="left")

print(f"Lignes avant : {len(titanic)} / après : {len(titanic_pays)}")
titanic_pays

Lignes avant : 891 / après : 891


,survived,pclass,sex,age,sibsp,parch,fare,embark_town,alone,pays
0,0,3,male,22.0,1,0,7.2500,Southampton,False,Angleterre
1,1,1,female,38.0,1,0,71.2833,Cherbourg,False,France
2,1,3,female,26.0,0,0,7.9250,Southampton,True,Angleterre
3,1,1,female,35.0,1,0,53.1000,Southampton,False,Angleterre
4,0,3,male,35.0,0,0,8.0500,Southampton,True,Angleterre
...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,Southampton,True,Angleterre
887,1,1,female,19.0,0,0,30.0000,Southampton,True,Angleterre
888,0,3,female,21.5,1,2,23.4500,Southampton,False,Angleterre
889,1,1,male,26.0,0,0,30.0000,Cherbourg,True,France


In [3]:
passagers_par_pays = titanic_pays["pays"].value_counts()
passagers_par_pays

pays
Angleterre    646
France        168
Irlande        77
Name: count, dtype: int64

In [4]:
survie_par_pays = (titanic_pays.groupby("pays")["survived"].mean() * 100).round(1)
survie_par_pays

pays
Angleterre    33.9
France        55.4
Irlande       39.0
Name: survived, dtype: float64

**Note :** l'Angleterre compte 646 passagers et non 644 : à l'exercice 08, les 2 passagers sans port ont été remplis par Southampton au lieu d'être supprimés. France (168) et Irlande (77) sont identiques à la vérification. Les passagers partis de France survivent le mieux (55.4 %), ceux d'Angleterre le moins (33.9 %).

In [5]:
ports_sans_queenstown = ports[ports["embark_town"] != "Queenstown"]

jointure_inner = titanic.merge(ports_sans_queenstown, on="embark_town", how="inner")
jointure_left = titanic.merge(ports_sans_queenstown, on="embark_town", how="left")

print(f"inner : {len(jointure_inner)} lignes")
print(f"left : {len(jointure_left)} lignes, pays manquants : {jointure_left['pays'].isna().sum()}")
jointure_left

inner : 814 lignes
left : 891 lignes, pays manquants : 77


,survived,pclass,sex,age,sibsp,parch,fare,embark_town,alone,pays
0,0,3,male,22.0,1,0,7.2500,Southampton,False,Angleterre
1,1,1,female,38.0,1,0,71.2833,Cherbourg,False,France
2,1,3,female,26.0,0,0,7.9250,Southampton,True,Angleterre
3,1,1,female,35.0,1,0,53.1000,Southampton,False,Angleterre
4,0,3,male,35.0,0,0,8.0500,Southampton,True,Angleterre
...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,Southampton,True,Angleterre
887,1,1,female,19.0,0,0,30.0000,Southampton,True,Angleterre
888,0,3,female,21.5,1,2,23.4500,Southampton,False,Angleterre
889,1,1,male,26.0,0,0,30.0000,Cherbourg,True,France


In [6]:
jointure_left[jointure_left["pays"].isna()]

,survived,pclass,sex,age,sibsp,parch,fare,embark_town,alone,pays
5,0,3,male,25.0,0,0,8.4583,Queenstown,True,NaN
16,0,3,male,2.0,4,1,29.1250,Queenstown,False,NaN
22,1,3,female,15.0,0,0,8.0292,Queenstown,True,NaN
28,1,3,female,21.5,0,0,7.8792,Queenstown,True,NaN
32,1,3,female,21.5,0,0,7.7500,Queenstown,True,NaN
...,...,...,...,...,...,...,...,...,...,...
790,0,3,male,25.0,0,0,7.7500,Queenstown,True,NaN
825,0,3,male,25.0,0,0,6.9500,Queenstown,True,NaN
828,1,3,male,25.0,0,0,7.7500,Queenstown,True,NaN
885,0,3,female,39.0,0,5,29.1250,Queenstown,False,NaN


**Observation :** une fois Queenstown retiré de la table `ports`, la jointure `inner` ne garde que les passagers qui ont une correspondance : 814 lignes, les 77 passagers de Queenstown disparaissent. La jointure `left` garde les 891 lignes, mais les 77 passagers de Queenstown ont un pays vide (NaN). Le `inner` perd donc des données sans prévenir, alors que le `left` les conserve en les marquant comme manquantes.